# CEVAE実証実験

潜在交絡因子（latent confounder）がある状況で、CEVAE がふつうの因果推論手法より ITE / CATE を
どれだけ正確に推定できるか、人工データで検証する。設計の詳細は `設計書.md` を参照。

このノートブックに実験の全コードをまとめている（DGP→評価指標→ベースライン→(今後)CEVAE→sweep→可視化）。


In [7]:
import warnings
import itertools
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy.stats import uniform, randint
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from xgboost import XGBRegressor
import torch
import torch.nn as nn

# macOS の Accelerate BLAS バックエンドが小さい行列の matmul で出す偽の警告を抑制
# （NaN/Inf が実際に出ていないことは確認済み。NumPy+Apple Silicon の既知の挙動）
warnings.filterwarnings("ignore", message=".*encountered in matmul.*", category=RuntimeWarning)


## 0. 一番簡単なデータ生成過程（理解の出発点）

複雑な設定（2次元のZ等）はまだ使わず、**Zは連続値（1個の数値、スカラー）、proxyは5本（X1~X5）、
ローディングは本ごとにランダムな係数**という構成で同じ仕組みを確認する。
ここだけは自己完結（後のセルの関数には依存しない）。

DAGは変わらない：
```
   Z（潜在交絡。観測しない）
  / | \
 X  T → Y
```

- `Z`：観測できない交絡因子。連続値（`N(0,1)`からサンプリングする1個の数値）。
- `X`：観測できる「Zの代理（proxy）」。5本あり、`X_j = a_j * Z + ノイズ_j`。
  `a_j` が「ローディング」＝proxy jがZをどれくらいの倍率で映しているかという係数。
  今回は **`a_j` を本ごとにランダムに決める**（`a_j ~ N(1.5, 1.5)`：平均1.5・標準偏差1.5）。
  平均を0からずらしているのは「共変量がマイナスの関係になることはあまり多くない」という考え方を反映したもの
  （ゼロにすると正負半々になりやすい。負になることも一応あるが、少数派になる）。
  `a_j` が大きい本ほどZをよく反映し、`a_j` が0に近い本はほぼノイズだけ（Zの情報をあまり持たない）。
- `T`：処置（0/1）。Zに依存させることで「交絡」を作る。
- `Y`：結果。Zに依存する部分（交絡）と、Tによる本当の効果の部分を足し合わせる。


In [8]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

rng = np.random.default_rng(0)
N = 20000
d = 5   # proxyの本数（X1~X5）

gamma = 1.5    # Z→T の交絡強度
theta = 2.0    # Z→Y の交絡強度
alpha = 1.0    # 真のATE
beta = 1.0     # 効果の異質性（Zの値によって効果がどれだけ変わるか）
sigma_x = 2.5  # proxyのノイズの大きさ（a_jが大きめなので、ノイズも上げて相関が高すぎないように調整）
sigma_y = 1.0  # 結果のノイズの大きさ（標準正規分布。Zと同じスケールにして分かりやすくする）

# ① 潜在交絡 Z（連続値。1サンプルにつき1個の数値）
Z = rng.normal(0, 1, size=N)

# ② proxy X_j = a_j*Z + ノイズ_j （j=1..5）。a_j は本ごとにランダムな係数（ローディング）
# 平均1.5・標準偏差1.5にして、分散を広げつつ正の値が多くなるようにする（負になることもあるが少数派）
a = rng.normal(1.5, 1.5, size=d)   # 例: [2.3, 0.4, -0.6, 3.1, 1.8] のようにバラバラ
X = Z[:, None] * a[None, :] + rng.normal(0, sigma_x, size=(N, d))  # shape (N, 5)

# ③ 処置 T ~ Bernoulli(sigmoid(gamma * Z))　← Zに依存して決まるので「交絡」が生まれる
T = rng.binomial(1, sigmoid(gamma * Z))

# ④ 結果 Y(t) = theta*Z + t*(alpha + beta*Z) + ノイズ
tau = alpha + beta * Z     # 真のITE（個体ごとの本当の処置効果）
base = theta * Z           # Zに依存するベース水準（これが交絡の原因）
Y0 = base + rng.normal(0, sigma_y, size=N)            # 処置を受けなかった場合の結果
Y1 = base + tau + rng.normal(0, sigma_y, size=N)  # 処置を受けた場合の結果
Y = np.where(T == 1, Y1, Y0)  # 実際に観測されるのはどちらか一方だけ


In [9]:
# 生成過程に関わる変数を最初の20人分だけ表で確認
cols = {"Z": Z}
cols.update({f"X{j+1}": X[:, j] for j in range(d)})
cols.update({"T": T, "Y0": Y0, "Y1": Y1, "Y": Y, "tau": tau})
df_preview = pd.DataFrame(cols)
df_preview.head(20)


,Z,X1,X2,X3,X4,X5,T,Y0,Y1,Y,tau
0,0.125730,-1.576204,4.227750,3.509423,2.643070,-3.133414,0,0.829781,0.853931,0.829781,1.125730
1,-0.132105,-0.183670,-2.058484,-0.015687,-1.142550,-1.234945,1,-1.344746,1.471250,1.471250,0.867895
2,0.640423,1.838766,2.896035,2.528364,4.561836,8.157933,1,2.171406,2.717749,2.717749,1.640423
3,0.104900,-0.218430,4.433979,-1.264502,-2.679743,0.129495,1,-1.538939,1.880420,1.880420,1.104900
4,-0.535669,0.706782,-3.278713,-3.553947,0.178341,-0.452694,0,-0.866805,0.247881,-0.866805,0.464331
5,0.361595,-2.253958,2.060840,2.042736,1.369853,1.554916,1,1.630369,0.886077,0.886077,1.361595
6,1.304000,3.462162,5.052939,4.284778,4.806470,5.578225,0,2.857618,5.174856,2.857618,2.304000
7,0.947081,0.651674,2.270553,-1.408584,0.859392,0.777116,1,1.569363,4.781167,4.781167,1.947081
8,-0.703735,-4.073484,-0.855576,-2.533810,2.538905,-0.295929,0,-1.764793,-1.524852,-1.764793,0.296265
9,-1.265421,-2.722941,-3.596839,-3.420310,0.520593,1.401066,0,-3.645318,-3.617181,-3.645318,-0.265421


In [10]:
true_ate = tau.mean()
naive_ate = Y[T == 1].mean() - Y[T == 0].mean()

corr_xz = np.array([np.corrcoef(X[:, j], Z)[0, 1] for j in range(d)])

print(f"真のATE                : {true_ate:.3f}")
print(f"ナイーブATE(差の平均)   : {naive_ate:.3f}   ← 交絡バイアス = {naive_ate - true_ate:+.3f}")
print(f"ローディング a_j ({d}本)  : {np.round(a, 3)}")
print(f"corr(X_j, Z) ({d}本)      : {np.round(corr_xz, 3)}  (|a_j|が大きい本ほど相関が強くなるはず)")
print(f"corr(T, Z)              : {np.corrcoef(T, Z)[0,1]:.3f}  (0なら交絡なし。今回はgamma=1.5なので非ゼロ)")


真のATE                : 1.005
ナイーブATE(差の平均)   : 3.651   ← 交絡バイアス = +2.646
ローディング a_j (5本)  : [1.985 3.19  1.706 2.365 1.876]
corr(X_j, Z) (5本)      : [0.616 0.783 0.566 0.691 0.604]  (|a_j|が大きい本ほど相関が強くなるはず)
corr(T, Z)              : 0.531  (0なら交絡なし。今回はgamma=1.5なので非ゼロ)


**読み方**：ナイーブATE（単純にT=1グループとT=0グループのYの平均差をとっただけの推定）は、
真のATE（=alpha）から大きくズレているはず。これが「交絡によるバイアス」そのもの。

`a_j` と `corr(X_j, Z)` を並べて見ると、**`|a_j|` が大きい本ほど `corr(X_j, Z)` の絶対値も大きい**という
対応関係が見えるはず（符号がマイナスなら相関もマイナスになる）。これが「ローディングがproxyの情報量を決める」ということの実例。

次のセクション以降（## 1. 以降）では、これと同じ仕組みを「Zを2次元・proxyを10本」に拡張したもの
（本実験用の設定。ローディングも同様にランダムに決めて固定する）を使う。


### X-learner（交絡を考慮できないver。去年の卒論コードがベース）

去年の卒論コード（S/T/X-learner一式）のうち、**X-learnerだけ**をこのセクションのデータに適用する。
今回はCEVAEに重きを置きたいので、比較対象はX-learner一本にする（Naive/S/T-learnerは省略）。
3つの中でX-learnerが一番洗練されている（傾向スコアで補正する）ので、CEVAEにとって一番厳しい比較相手になる。

やることは去年と同じ3段階：
1. T-learner（T=0/1で別モデルを学習）をまず作る（X-learnerの内部ステップとして必要）
2. その2つのモデルから「疑似ITE」（D0, D1）を計算し、それをXで回帰するモデルを作る（→ tau0, tau1）
3. 傾向スコア（LogisticRegression）で tau0 と tau1 を合成する

X-learnerが見るのは `X`（=Zの代理。proxy）と `T`, `Y` だけ。**Zそのものは絶対に渡さない**ので、
ここでの予測精度には「proxyの不完全さ」による残差交絡が残るはず。


In [11]:
proxy_cols = [f"X{j+1}" for j in range(d)]
df_full = pd.DataFrame(X, columns=proxy_cols)
df_full["T"] = T
df_full["Y"] = Y
df_full["tau"] = tau

df_train, df_test = train_test_split(df_full, test_size=0.3, random_state=0, stratify=df_full["T"])

XGB_PARAM_DIST = {
    "n_estimators": randint(300, 2500),
    "learning_rate": uniform(0.01, 0.19),
    "max_depth": randint(2, 9),
    "min_child_weight": randint(1, 10),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "reg_lambda": uniform(0.0, 10.0),
}

def search_xgb_params(X_in, y_in, n_iter=50, cv=5, seed=0):
    """RandomizedSearchCVでXGBRegressorの最良パラメータを探索する（去年の卒論コードと同じ設定）。"""
    search = RandomizedSearchCV(
        XGBRegressor(objective="reg:squarederror", random_state=seed, verbosity=0),
        param_distributions=XGB_PARAM_DIST,
        n_iter=n_iter,
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
        random_state=seed,
    )
    search.fit(X_in, y_in)
    return search.best_estimator_.get_params()


In [12]:
df0 = df_train[df_train["T"] == 0]
df1 = df_train[df_train["T"] == 1]

# ① T-learner（X-learnerの内部ステップとして必要）
t0_params = search_xgb_params(df0[proxy_cols], df0["Y"])
t1_params = search_xgb_params(df1[proxy_cols], df1["Y"])
t0_model = XGBRegressor(**t0_params).fit(df0[proxy_cols], df0["Y"])
t1_model = XGBRegressor(**t1_params).fit(df1[proxy_cols], df1["Y"])

# ② 疑似ITE（D0, D1）を計算してXで回帰
D0 = t1_model.predict(df0[proxy_cols]) - df0["Y"].values
D1 = df1["Y"].values - t0_model.predict(df1[proxy_cols])
x0_params = search_xgb_params(df0[proxy_cols], D0)
x1_params = search_xgb_params(df1[proxy_cols], D1)
x0_model = XGBRegressor(**x0_params).fit(df0[proxy_cols], D0)
x1_model = XGBRegressor(**x1_params).fit(df1[proxy_cols], D1)

# ③ 傾向スコアで tau0, tau1 を合成
prop_model = LogisticRegression(max_iter=1000).fit(df_train[proxy_cols], df_train["T"])
g0_test = prop_model.predict_proba(df_test[proxy_cols])[:, 0]   # P(T=0|x)
tau0_test = x0_model.predict(df_test[proxy_cols])
tau1_test = x1_model.predict(df_test[proxy_cols])
tau_pred_x = g0_test * tau0_test + (1 - g0_test) * tau1_test


/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [13]:
df_test

,X1,X2,X3,X4,X5,T,Y,tau
15540,-8.632553,-4.399203,-0.588903,-0.684505,-4.060644,0,-2.333408,-0.077633
17522,-3.129181,0.492861,1.919884,-2.312849,2.218690,0,0.127012,0.767613
6459,6.287738,0.698256,2.948731,0.651257,0.633898,1,4.651362,2.315422
10472,4.500469,-0.667036,-0.572471,3.200098,-1.030310,0,-0.768721,0.703813
4212,4.544263,1.107476,-1.196626,-1.506096,0.155359,1,4.895535,2.118968
...,...,...,...,...,...,...,...,...
15471,3.551796,0.130468,-2.179031,4.936625,3.586389,1,2.013410,1.843160
10188,-1.821735,-3.378120,0.116293,-4.302244,-3.769951,0,-0.717900,0.333608
1937,-2.977982,1.482913,3.114003,-2.643034,0.727742,1,0.882825,1.224803
5955,2.173642,-2.354309,-2.881396,-1.283939,2.022454,1,1.908958,1.443580


In [14]:
tau_true_test = df_test["tau"].values
sqrt_pehe_x = np.sqrt(np.mean((tau_true_test - tau_pred_x) ** 2))
mae_x = np.mean(np.abs(tau_true_test - tau_pred_x))
eps_ate_x = abs(np.mean(tau_pred_x) - np.mean(tau_true_test))

print(f"X-learner  \u221aPEHE={sqrt_pehe_x:.3f}  MAE={mae_x:.3f}  eps_ATE={eps_ate_x:.3f}")


X-learner  √PEHE=0.844  MAE=0.708  eps_ATE=0.661


## CEVAE Part 1：生成モデル（Decoder）の3ネットワーク

CEVAEは「Zさえ分かれば、X・T・Yが生成できる」というモデルを学習する。
Part 1では、その「生成する側」（Decoder）の3つのネットワークだけを作る（まだ学習はしない・ランダムな重みのまま）。

設計書§4.2の対応：
```
p(z)      = N(0, I)            ← 既にDGPで使っているのでここでは何もしない
p(x | z)  : MLP(z) → proxy(X1~X5)の平均（ガウス。分散は固定とみなす）
p(t | z)  : MLP(z) → 処置TのBernoulliロジット
p(y | t,z): t=0用・t=1用で別ヘッド（TARNet構造）→ 結果の平均
```

潜在変数`z`の次元は、今回は**真のZもスカラー（1次元）なので、CEVAE側も z_dim=1 に合わせる**
（本実験の2次元版に進むときはここを2に変える）。


In [ ]:
torch.manual_seed(0)

Z_DIM = 1     # CEVAEの潜在変数zの次元（真のZがスカラーなので、まずはz_dim=1に合わせる）
HIDDEN = 32   # 隠れ層のユニット数（proxy5本・N=20000くらいの小さい問題なので、小さめでよい）


class DecoderX(nn.Module):
    """p(x|z)：zからproxy X1~X5の平均を予測する（ガウス分布、分散は固定とみなす）"""
    def __init__(self, z_dim=Z_DIM, d=d, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, d),
        )

    def forward(self, z):
        return self.net(z)          # (batch, d) Xの平均


class DecoderT(nn.Module):
    """p(t|z)：zから処置Tのロジットを予測する（Bernoulli）"""
    def __init__(self, z_dim=Z_DIM, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, z):
        return self.net(z).squeeze(-1)   # (batch,) ロジット（sigmoidを通す前の値）


class DecoderY(nn.Module):
    """p(y|t,z)：TARNet構造。t=0用・t=1用で完全に別のヘッド（ネットワーク）を持つ"""
    def __init__(self, z_dim=Z_DIM, hidden=HIDDEN):
        super().__init__()
        self.head0 = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.head1 = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, z):
        y0_hat = self.head0(z).squeeze(-1)   # p(y|t=0, z) の平均
        y1_hat = self.head1(z).squeeze(-1)   # p(y|t=1, z) の平均
        return y0_hat, y1_hat


decoder_x = DecoderX()
decoder_t = DecoderT()
decoder_y = DecoderY()


**動作確認**：今はまだ学習していない（ランダムな重みのまま）ので、出てくる数字自体に意味はない。
ここで確認したいのは「形（shape）が合っているか」と「とりあえずエラーなく動くか」だけ。

真のZを使ってforwardしてみる（本来はCEVAEはZを直接見られないが、Decoder単体の動作確認のためだけに今回は使う）。


In [ ]:
z_true_tensor = torch.tensor(Z, dtype=torch.float32).unsqueeze(1)   # (N, 1)

with torch.no_grad():
    x_hat = decoder_x(z_true_tensor)
    t_logit = decoder_t(z_true_tensor)
    y0_hat, y1_hat = decoder_y(z_true_tensor)

print("x_hat   shape:", tuple(x_hat.shape), "  (期待: (N, d) = (", N, ",", d, "))")
print("t_logit shape:", tuple(t_logit.shape), "  (期待: (N,) = (", N, ",))")
print("y0_hat  shape:", tuple(y0_hat.shape), " y1_hat shape:", tuple(y1_hat.shape))

print()
print("最初の5人分の x_hat（未学習なので、まだランダムな値）:")
print(x_hat[:5].detach().numpy().round(3))


## CEVAE Part 2：推論モデル（Encoder）の3ネットワーク

Part 1では「Zが分かればX・T・Yを生成できる」側（Decoder）を作った。
だがCEVAEはZを直接観測できないので、推論時には逆向き、**「観測したX・T・Yから、Zはどんな分布だったかを推定する」**
側（Encoder, q）が必要になる。Part 2では、その3つのネットワークを作る（まだ学習はしない・ランダムな重みのまま）。

設計書§4.1の対応：
```
q(t | x)         : MLP(x) → Bernoulli logit                     # 補助NW
q(y | x, t)      : 2ヘッド MLP（t=0/t=1 で別ヘッド）→ 平均        # 補助NW
q(z | x, t, y)   : 共有表現 h = MLP(x);  t で分岐し [h, y] → (μ_z, logσ²_z)
                   → z ~ N(μ_z, σ²_z)（reparameterization）
```

`q(t|x)` と `q(y|x,t)` は「補助ネットワーク（auxiliary network）」と呼ばれるもので、
テスト時に「xだけ」からt・yを予測するために使う（設計書§4.4のステップ1・2）。
`q(z|x,t,y)` が本体の推論ネットワークで、学習時はx・t・y全部が見える状況でzの事後分布を近似する。
Part 1と同じく、まずはPart 0のトイデータ（Zはスカラー、proxyはX1~X5）に合わせて `z_dim=1` で作る。

In [ ]:
class EncoderT(nn.Module):
    """q(t|x)：xから処置Tのロジットを予測する補助ネットワーク（テスト時にtを推定するために使う）"""
    def __init__(self, d=d, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)   # (batch,) ロジット


class EncoderY(nn.Module):
    """q(y|x,t)：TARNet構造の補助ネットワーク。t=0用・t=1用で別ヘッドの平均を出す
    （実際にどちらを使うかはtで選ぶ。テスト時にyを推定するために使う）"""
    def __init__(self, d=d, hidden=HIDDEN):
        super().__init__()
        self.head0 = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.head1 = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x):
        y0_hat = self.head0(x).squeeze(-1)   # q(y|x, t=0) の平均
        y1_hat = self.head1(x).squeeze(-1)   # q(y|x, t=1) の平均
        return y0_hat, y1_hat


class EncoderZ(nn.Module):
    """q(z|x,t,y)：本体の推論ネットワーク。
    まずxだけから共有表現hを作り（trunk）、tで枝分かれした先で[h, y]からzの平均・log分散を出す。
    実際に使う枝（t=0用 / t=1用）は観測されたtで選ぶ。
    """
    def __init__(self, d=d, z_dim=Z_DIM, hidden=HIDDEN):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(d, hidden), nn.ReLU())
        self.head0 = nn.Sequential(nn.Linear(hidden + 1, hidden), nn.ReLU())
        self.head1 = nn.Sequential(nn.Linear(hidden + 1, hidden), nn.ReLU())
        self.mu0 = nn.Linear(hidden, z_dim)
        self.logvar0 = nn.Linear(hidden, z_dim)
        self.mu1 = nn.Linear(hidden, z_dim)
        self.logvar1 = nn.Linear(hidden, z_dim)

    def forward(self, x, t, y):
        h = self.trunk(x)                             # (batch, hidden) ← xだけから作る共有表現
        hy = torch.cat([h, y.unsqueeze(-1)], dim=-1)   # (batch, hidden+1) ← yを連結
        h0 = self.head0(hy)
        h1 = self.head1(hy)
        mu0, logvar0 = self.mu0(h0), self.logvar0(h0)
        mu1, logvar1 = self.mu1(h1), self.logvar1(h1)
        t_col = t.unsqueeze(-1)                        # (batch, 1) ← (batch, z_dim)とブロードキャストするため
        mu = torch.where(t_col == 1, mu1, mu0)
        logvar = torch.where(t_col == 1, logvar1, logvar0)
        return mu, logvar


def reparameterize(mu, logvar):
    """reparameterization trick：z = mu + eps*std（epsは標準正規からサンプル）。
    勾配がサンプリングを通り抜けられるようにするためのCEVAE/VAEの定石。
    """
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std


encoder_t = EncoderT()
encoder_y = EncoderY()
encoder_z = EncoderZ()

**動作確認**：Part 1と同様、まだ学習していないのでzの推定値そのものに意味はない。
ここでは「形（shape）が合っているか」「エラーなく動くか」だけを確認する。

今度はDecoderと逆で、観測できる `X, T, Y`（Part 0で作ったトイデータ）を入力にして forward する。
本来のCEVAEの学習ループでは、ここで出た `mu_z, logvar_z` を使ってELBOのKL項を計算し、
`reparameterize` でサンプルした `z` をPart 1のDecoderに渡して再構成誤差を計算する（Part 3予定）。

In [ ]:
x_tensor = torch.tensor(X, dtype=torch.float32)        # (N, d)
t_tensor = torch.tensor(T, dtype=torch.float32)         # (N,)
y_tensor = torch.tensor(Y, dtype=torch.float32)          # (N,)

with torch.no_grad():
    t_logit_q = encoder_t(x_tensor)
    y0_hat_q, y1_hat_q = encoder_y(x_tensor)
    mu_z, logvar_z = encoder_z(x_tensor, t_tensor, y_tensor)
    z_sample = reparameterize(mu_z, logvar_z)

print("t_logit_q shape:", tuple(t_logit_q.shape), "  (期待: (N,) = (", N, ",))")
print("y0_hat_q  shape:", tuple(y0_hat_q.shape), " y1_hat_q shape:", tuple(y1_hat_q.shape))
print("mu_z      shape:", tuple(mu_z.shape), "  (期待: (N, z_dim) = (", N, ",", Z_DIM, "))")
print("logvar_z  shape:", tuple(logvar_z.shape))
print("z_sample  shape:", tuple(z_sample.shape))

print()
print("最初の5人分の mu_z（未学習なので、まだランダムな値）:")
print(mu_z[:5].detach().numpy().round(3))

## CEVAE Part 3：損失関数（−ELBO + 補助損失）

Part 1（Decoder, p）とPart 2（Encoder, q）が揃ったので、この2つを繋いで
1回のforwardで損失（学習で最小化したい量）を計算できるようにする。まだoptimizerは作らず、
「損失が計算できて、backward()で勾配が流れるか」だけを確認する（学習ループはPart 4予定）。

設計書§4.3の対応：
```
L = 再構成_x (Gaussian NLL ≒ MSE)
  + 再構成_t (BCE)
  + 再構成_y (MSE)
  + KL( q(z|x,t,y) || N(0,I) )          # ← β_kl で重み付け（KLアニーリング）
  + 補助: BCE(q(t|x), t) + MSE(q(y|x,t), y)
```

流れ（学習時は t, y が両方観測されている前提）：
1. `encoder_z(x, t, y)` で事後分布 q(z|x,t,y) の `mu_z, logvar_z` を求める
2. `reparameterize` で `z` をサンプル
3. その `z` を Part 1 の3つの Decoder に通し、x・t・y を再構成できているかを損失にする（reconstruction）
4. `q(z|x,t,y)` と事前分布 `N(0,I)` のKLダイバージェンスを計算する（正則化）
5. 加えて、テスト時に使う補助ネットワーク `encoder_t(x)` / `encoder_y(x)` 自体もこの段階で学習させておく
   （xだけからt・yを当てる練習。設計書§4.4のテスト時推定で使うのはこちら）

**KLアニーリング**：学習初期からKL項をフルで効かせると、Encoderが「zを使わずに済ませよう」として
z の情報がDecoderに伝わらなくなる現象（posterior collapse）が起きやすい。そこで `beta_kl` を
0から1へ徐々に上げる重みとして外から渡せるようにしておく（学習ループ側でstepごとに更新する想定）。

In [ ]:
def kl_to_standard_normal(mu, logvar):
    """q(z|x,t,y) = N(mu, exp(logvar)) と事前分布 N(0,I) のKLダイバージェンス（解析解）。
    batch平均を返す（z_dim方向はsumしてからbatch方向にmeanする）。
    """
    kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())   # (batch, z_dim)
    return kl_per_dim.sum(dim=-1).mean()


def linear_kl_anneal(step: int, total_steps: int) -> float:
    """KLアニーリングの重み beta_kl を 0→1 へ線形に上げるスケジュール。
    学習ループ側で `beta_kl = linear_kl_anneal(現在のstep, アニーリング終了までのstep数)` のように使う想定。
    """
    if total_steps <= 0:
        return 1.0
    return min(1.0, step / total_steps)


def cevae_loss(x, t, y, beta_kl=1.0):
    """CEVAEの1バッチ分の損失（−ELBO + 補助損失）を計算する。
    x: (batch, d)  t: (batch,) 0/1  y: (batch,)
    decoder_x/decoder_t/decoder_y（Part 1）と encoder_t/encoder_y/encoder_z（Part 2）を使う。
    戻り値: (total_loss, 各項のdict（ログ用、勾配は持たないfloatに変換済み）)
    """
    # 1) 事後分布 q(z|x,t,y) から z をサンプル
    mu_z, logvar_z = encoder_z(x, t, y)
    z = reparameterize(mu_z, logvar_z)

    # 2) Decoder で x, t, y を再構成
    x_hat = decoder_x(z)
    t_logit_p = decoder_t(z)
    y0_hat_p, y1_hat_p = decoder_y(z)
    y_hat_p = torch.where(t == 1, y1_hat_p, y0_hat_p)

    recon_x = nn.functional.mse_loss(x_hat, x)
    recon_t = nn.functional.binary_cross_entropy_with_logits(t_logit_p, t)
    recon_y = nn.functional.mse_loss(y_hat_p, y)

    # 3) KL( q(z|x,t,y) || N(0,I) )
    kl = kl_to_standard_normal(mu_z, logvar_z)

    # 4) 補助損失：テスト時にxだけからt・yを当てるための補助ネットワークもここで学習させる
    t_logit_q = encoder_t(x)
    y0_hat_q, y1_hat_q = encoder_y(x)
    y_hat_q = torch.where(t == 1, y1_hat_q, y0_hat_q)

    aux_t = nn.functional.binary_cross_entropy_with_logits(t_logit_q, t)
    aux_y = nn.functional.mse_loss(y_hat_q, y)

    total = recon_x + recon_t + recon_y + beta_kl * kl + aux_t + aux_y

    components = {
        "total": total.item(),
        "recon_x": recon_x.item(),
        "recon_t": recon_t.item(),
        "recon_y": recon_y.item(),
        "kl": kl.item(),
        "aux_t": aux_t.item(),
        "aux_y": aux_y.item(),
    }
    return total, components

**動作確認**：損失が計算できること、`backward()` で全ネットワーク（Decoder3つ・Encoder3つ）に
勾配が流れることを確認する。まだ1回もoptimizer.step()していないので、損失の値そのものに意味はない。

In [ ]:
loss, components = cevae_loss(x_tensor, t_tensor, y_tensor, beta_kl=1.0)
loss.backward()

print("損失の内訳:")
for k, v in components.items():
    print(f"  {k:8s}: {v:.4f}")

print()
print("勾配が流れているか確認（None でなければOK）:")
for name, net in [("decoder_x", decoder_x), ("decoder_t", decoder_t), ("decoder_y", decoder_y),
                   ("encoder_t", encoder_t), ("encoder_y", encoder_y), ("encoder_z", encoder_z)]:
    has_grad = all(p.grad is not None for p in net.parameters())
    print(f"  {name:10s}: {'OK' if has_grad else 'NG（勾配なし）'}")

print()
print("KLアニーリングのスケジュール例（total_steps=1000）:")
print([round(linear_kl_anneal(s, 1000), 2) for s in [0, 100, 250, 500, 750, 1000]])

## CEVAE Part 4：学習ループ（optimizerで実際に学習する）

Part 1〜3で「損失が計算でき、backward()で勾配が流れる」ことまで確認した。
Part 4ではいよいよ `optimizer.step()` を呼んで重みを更新し、実際に学習させる。

まずは**Part 0のトイデータ（Zはスカラー、proxyはX1~X5、z_dim=1）で1回通して試す**
（本番の2次元DGPへの切り替えは後の課題）。

1ステップの中身は：
```
optimizer.zero_grad()                 # 前stepの勾配をリセット
loss, components = cevae_loss(...)    # forward（beta_klはKLアニーリングで変える）
loss.backward()                       # 勾配を計算
optimizer.step()                      # 勾配の向きにパラメータを更新
```
これをミニバッチ単位で何epochも繰り返す。`beta_kl` は `linear_kl_anneal`（Part 3）で
epochが進むごとに0→1へ上げる。

In [ ]:
torch.manual_seed(0)

EPOCHS = 200
BATCH_SIZE = 256
LR = 1e-3
KL_ANNEAL_EPOCHS = 100   # この epoch 数かけて beta_kl を 0→1 へ線形に上げる

all_params = (
    list(decoder_x.parameters()) + list(decoder_t.parameters()) + list(decoder_y.parameters())
    + list(encoder_t.parameters()) + list(encoder_y.parameters()) + list(encoder_z.parameters())
)
optimizer = torch.optim.Adam(all_params, lr=LR)

n_batches = N // BATCH_SIZE
history = []

for epoch in range(EPOCHS):
    beta_kl = linear_kl_anneal(epoch, KL_ANNEAL_EPOCHS)
    perm = torch.randperm(N)
    epoch_sums = {"total": 0.0, "recon_x": 0.0, "recon_t": 0.0, "recon_y": 0.0, "kl": 0.0, "aux_t": 0.0, "aux_y": 0.0}

    for b in range(n_batches):
        idx = perm[b * BATCH_SIZE: (b + 1) * BATCH_SIZE]
        x_batch, t_batch, y_batch = x_tensor[idx], t_tensor[idx], y_tensor[idx]

        optimizer.zero_grad()
        loss, components = cevae_loss(x_batch, t_batch, y_batch, beta_kl=beta_kl)
        loss.backward()
        optimizer.step()

        for k in epoch_sums:
            epoch_sums[k] += components[k]

    epoch_avg = {k: v / n_batches for k, v in epoch_sums.items()}
    epoch_avg["beta_kl"] = beta_kl
    history.append(epoch_avg)

    if epoch % 20 == 0 or epoch == EPOCHS - 1:
        c = epoch_avg
        print(f"epoch {epoch:4d}  beta_kl={c['beta_kl']:.2f}  total={c['total']:.3f}  "
              f"recon_x={c['recon_x']:.3f} recon_t={c['recon_t']:.3f} recon_y={c['recon_y']:.3f} "
              f"kl={c['kl']:.3f} aux_t={c['aux_t']:.3f} aux_y={c['aux_y']:.3f}")

**動作確認**：学習が進んで損失（特に `recon_x` / `recon_y`）が下がっているか確認する。
さらにトイデータでは真の `Z` を知っているので、学習後の `encoder_z` がどれだけ `Z` を
復元できているか（`corr(推定z, 真のZ)`）も見る。これがCEVAEの「Zを復元できているか」の
一番シンプルな成功指標になる（設計書§7の「復元したz vs 真のZ」図の元データ）。

In [ ]:
with torch.no_grad():
    mu_z_trained, _ = encoder_z(x_tensor, t_tensor, y_tensor)

corr_z = np.corrcoef(mu_z_trained.squeeze(-1).numpy(), Z)[0, 1]
print(f"corr(推定z, 真のZ) = {corr_z:.3f}  (絶対値が1に近いほど、CEVAEがZをうまく復元できている)")
print(f"（参考）学習開始時 total loss: {history[0]['total']:.3f}  →  学習終了時: {history[-1]['total']:.3f}")

## CEVAE Part 5：ITE推定（テスト時の手順、設計書§4.4）

学習済みのEncoder/Decoderを使い、**xだけ**からITEを推定する。テスト時はt・yを見られない前提なので、
設計書§4.4の手順通り、まず補助NWでt・yを「予測」してから、それを使ってzの事後分布を求める：
```
1. t̂ = q(t|x)                         # 補助NW（encoder_t）で処置を予測
2. ŷ = q(y|x, t̂)                       # 補助NW（encoder_y）で結果を予測
3. z ~ q(z|x, t̂, ŷ)（ここでは μ_z をそのまま点推定として使う）
4. y0 = p(y|t=0, z),  y1 = p(y|t=1, z)  # Decoder（decoder_y）
5. ITE_pred = y1 - y0
```

評価には、X-learnerの比較で使った `df_test`（同じトイデータから作った6000件のテスト分割、
`X1~X5, T, Y, tau` を持つ）をそのまま使う。

**注意（フェアな比較ではない点）**：Part 4の学習は `df_train/df_test` に分ける前の **N=20000全部**を
使って行った（トイデータでの動作確認が目的だったため）。つまりこの `df_test` の行もCEVAEは学習時に
見てしまっている。X-learner側は `df_train` だけで学習しているので、このままだとCEVAEに有利な
比較になってしまう。本番の2次元DGPに切り替えるときは、学習も `df_train` だけで行うように直す。

In [ ]:
def estimate_ite(x: torch.Tensor) -> torch.Tensor:
    """設計書§4.4の手順で、x（proxyのみ）からITEを推定する。学習済みのEncoder/Decoderを使う。"""
    with torch.no_grad():
        t_prob = torch.sigmoid(encoder_t(x))
        t_hat = (t_prob > 0.5).float()                     # 1) q(t|x) を2値化

        y0_hat_q, y1_hat_q = encoder_y(x)
        y_hat = torch.where(t_hat == 1, y1_hat_q, y0_hat_q)  # 2) q(y|x, t̂)

        mu_z, _ = encoder_z(x, t_hat, y_hat)                # 3) q(z|x, t̂, ŷ) の平均をそのまま使う

        y0_p, y1_p = decoder_y(mu_z)                        # 4) p(y|t=0,z), p(y|t=1,z)
        ite_pred = y1_p - y0_p                               # 5) ITE_pred
    return ite_pred


x_test_tensor = torch.tensor(df_test[proxy_cols].values, dtype=torch.float32)
tau_pred_cevae = estimate_ite(x_test_tensor).numpy()

df_test_cevae = df_test.copy()
df_test_cevae["tau_pred_cevae"] = tau_pred_cevae
df_test_cevae

In [ ]:
tau_true_test = df_test["tau"].values
sqrt_pehe_cevae = np.sqrt(np.mean((tau_true_test - tau_pred_cevae) ** 2))
mae_cevae = np.mean(np.abs(tau_true_test - tau_pred_cevae))
eps_ate_cevae = abs(np.mean(tau_pred_cevae) - np.mean(tau_true_test))

print(f"CEVAE      √PEHE={sqrt_pehe_cevae:.3f}  MAE={mae_cevae:.3f}  eps_ATE={eps_ate_cevae:.3f}")
print(f"X-learner  √PEHE={sqrt_pehe_x:.3f}  MAE={mae_x:.3f}  eps_ATE={eps_ate_x:.3f}")

**結果メモ（1次元トイデータ・初回実行）**：CEVAE √PEHE ≈ 15.3、X-learner √PEHE ≈ 0.84 で、CEVAEが大幅に負けた。

原因を `decoder_y` の `head0`/`head1` をzの関数として可視化して特定：z=0付近では妥当な値を出すが、
少し離れただけで激しく発散する（例：head0(z=-3)=-33.6、期待値は-6程度）。

理由は構造的なもの：T はZに依存して決まる（γ=1.5の交絡）ため、`head0`（t=0用）は
学習時にzのある偏った範囲しか見ておらず、`head1`（t=1用）は別の偏った範囲しか見ていない。
そのため反実仮想（counterfactual）側を予測するとき、各headは訓練データの外側に外挿することになり、
小さい2層ReLUネット（hidden=32, 正則化なし）がそこで暴れる。コードのバグではなく、
設計書§10で懸念されていた「CEVAEは学習が不安定」がそのまま表面化した形。

**次回試すべき対策（未着手）**：
- `decoder_y` / `decoder_x` / `decoder_t` への weight decay（L2正則化）
- 学習epoch数を増やす・学習率スケジュールを入れる
- 本番の2次元DGP（`DGPConfig`）に切り替え、`df_train` だけで学習するように直す（現状はN=20000全部で学習しておりtrain/testの分離もできていない）
- 反実仮想側のheadの出力を、観測データの z 範囲内にクリップする等の安定化


## CEVAE 診断1：学習後の `mu_z` は真の `Z` をどれだけ復元できているか

「CEVAEが負けている理由」を切り分けるための最初の一歩。**zそのものを掴めていないのか**、
**zは掴めているのにyやtauの推定が悪いのか**を区別したい。まずは前者（z復元度）を直接見る。

VAE系は `z` の符号・スケールが一意に定まらない（事前分布 `N(0,I)` は符号反転しても対称なので、
encoderが `+Z` 方向に学習するか `-Z` 方向に学習するかはランダム）。そのため `corr(mu_z, Z)` と
`corr(-mu_z, Z)` の両方を見て、**絶対値が大きい方**を実質的な復元度として読む。

Part4で学習済みの `encoder_z` をそのまま使う（trainデータ全部で学習したモデルなので、
ここではまだ train/test を分けていない点に注意。test分離した版はこの後のPart6以降で扱う）。

In [ ]:
with torch.no_grad():
    mu_z_check, _ = encoder_z(x_tensor, t_tensor, y_tensor)

mu_z_flat = mu_z_check.squeeze(-1).numpy()   # z_dim=1なので1列だけ
corr_pos = np.corrcoef(mu_z_flat, Z)[0, 1]
corr_neg = np.corrcoef(-mu_z_flat, Z)[0, 1]

print(f"corr(mu_z, Z)  = {corr_pos:+.3f}")
print(f"corr(-mu_z, Z) = {corr_neg:+.3f}")
print(f"→ 符号を考慮した実質的な復元度（絶対値最大の方） = {max(abs(corr_pos), abs(corr_neg)):.3f}")
print()
print("読み方の目安：")
print("  |corr| が 0.8 以上 … zはかなり良く復元できている → CEVAEが負けるならy/tau推定側の問題")
print("  |corr| が 0.3〜0.7 … 部分的にしか復元できていない → z推定の改善余地が大きい")
print("  |corr| が 0.2 未満 … ほぼ復元できていない（z推定がボトルネック）")


## CEVAE Part 6：ハイパーパラメータを変えられるようにリファクタリング

Part1〜5は「理解のための固定構成（z_dim=1, hidden=32, beta_kl最大1.0, N=20000全部で学習）」だった。
ここからは `z_dim` / `hidden` / `beta_kl` の最大値 / KLアニーリング期間 / `recon_y`・`aux_y` の重みを
外から指定できるように関数化し、**Part7のグリッドサーチで使い回せる形**にする。

あわせて、Part5の「結果メモ」で指摘した問題（**学習にtrain/test分割前の全データを使っていた**）も
ここで修正する：以後は `df_train` だけで学習し、`df_test` で評価する（X-learnerと同じ条件に揃える）。

追加した知見ベースの改善ノブ：
- `w_recon_y` / `w_aux_y`：再構成損失・補助損失のうちyに関する項の重み（recon_x/recon_t/aux_tに対する相対的な強さを調整できる）
- `weight_decay`：Adamのweight decay（L2正則化）。Part5の「結果メモ」で特定した
  「decoder_yのTARNetヘッドが学習データ範囲外で激しく外挿する」問題に直接効く可能性があるノブ
  （デフォルト0.0=今までと同じ挙動。Part7のグリッドサーチ後、必要なら振ってみる）。

In [ ]:
@dataclass
class CEVAEConfig:
    """CEVAEの構成・学習ハイパーパラメータ一式。Part1〜5で固定だった値を全部ここに集約する。"""
    z_dim: int = 1
    hidden: int = 32
    beta_kl_max: float = 1.0       # KLアニーリングが到達する最大の重み（Part1〜5は暗黙に1.0だった）
    kl_anneal_epochs: int = 75     # この epoch 数かけて beta_kl を 0→beta_kl_max へ上げる
    w_recon_y: float = 1.0         # 再構成損失のうちyの項の重み
    w_aux_y: float = 1.0           # 補助損失のうちyの項の重み
    epochs: int = 150
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 0.0      # Adamのweight decay（L2正則化）。0.0なら今までと同じ
    seed: int = 0


def build_cevae(cfg: CEVAEConfig, d: int) -> dict:
    """cfg.z_dim, cfg.hidden に従って、Decoder3つ・Encoder3つを毎回ゼロから新規作成する。
    Part1/Part2で定義したクラス（DecoderX/T/Y, EncoderT/Y/Z）はもともとz_dim/hiddenを
    引数で受け取れる作りなので、そのまま再利用できる。
    """
    torch.manual_seed(cfg.seed)
    return {
        "decoder_x": DecoderX(z_dim=cfg.z_dim, d=d, hidden=cfg.hidden),
        "decoder_t": DecoderT(z_dim=cfg.z_dim, hidden=cfg.hidden),
        "decoder_y": DecoderY(z_dim=cfg.z_dim, hidden=cfg.hidden),
        "encoder_t": EncoderT(d=d, hidden=cfg.hidden),
        "encoder_y": EncoderY(d=d, hidden=cfg.hidden),
        "encoder_z": EncoderZ(d=d, z_dim=cfg.z_dim, hidden=cfg.hidden),
    }


def cevae_loss_v2(nets: dict, x, t, y, beta_kl=1.0, w_recon_y=1.0, w_aux_y=1.0):
    """Part3の cevae_loss を一般化したもの。グローバル変数ではなく nets 辞書を受け取り、
    recon_y・aux_y の重みを外から指定できるようにした。中身のロジックはPart3と同じ。
    """
    mu_z, logvar_z = nets["encoder_z"](x, t, y)
    z = reparameterize(mu_z, logvar_z)

    x_hat = nets["decoder_x"](z)
    t_logit_p = nets["decoder_t"](z)
    y0_hat_p, y1_hat_p = nets["decoder_y"](z)
    y_hat_p = torch.where(t == 1, y1_hat_p, y0_hat_p)

    recon_x = nn.functional.mse_loss(x_hat, x)
    recon_t = nn.functional.binary_cross_entropy_with_logits(t_logit_p, t)
    recon_y = nn.functional.mse_loss(y_hat_p, y)

    kl = kl_to_standard_normal(mu_z, logvar_z)

    t_logit_q = nets["encoder_t"](x)
    y0_hat_q, y1_hat_q = nets["encoder_y"](x)
    y_hat_q = torch.where(t == 1, y1_hat_q, y0_hat_q)

    aux_t = nn.functional.binary_cross_entropy_with_logits(t_logit_q, t)
    aux_y = nn.functional.mse_loss(y_hat_q, y)

    total = recon_x + recon_t + w_recon_y * recon_y + beta_kl * kl + aux_t + w_aux_y * aux_y

    components = {
        "total": total.item(), "recon_x": recon_x.item(), "recon_t": recon_t.item(),
        "recon_y": recon_y.item(), "kl": kl.item(), "aux_t": aux_t.item(), "aux_y": aux_y.item(),
    }
    return total, components


def train_cevae(nets: dict, cfg: CEVAEConfig, x_tr, t_tr, y_tr, verbose_every: int = 0):
    """Part4の学習ループを一般化したもの。nets内の全パラメータをAdam1つでまとめて最適化する。
    戻り値: epochごとの損失内訳（list of dict）。verbose_every>0なら進捗を print する。
    """
    torch.manual_seed(cfg.seed)
    all_params = [p for net in nets.values() for p in net.parameters()]
    optimizer = torch.optim.Adam(all_params, lr=cfg.lr, weight_decay=cfg.weight_decay)

    n_samples = x_tr.shape[0]
    n_batches = max(1, n_samples // cfg.batch_size)
    history = []

    for epoch in range(cfg.epochs):
        beta_kl = cfg.beta_kl_max * linear_kl_anneal(epoch, cfg.kl_anneal_epochs)
        perm = torch.randperm(n_samples)
        epoch_sums = {"total": 0.0, "recon_x": 0.0, "recon_t": 0.0, "recon_y": 0.0, "kl": 0.0, "aux_t": 0.0, "aux_y": 0.0}

        for b in range(n_batches):
            idx = perm[b * cfg.batch_size:(b + 1) * cfg.batch_size]
            x_batch, t_batch, y_batch = x_tr[idx], t_tr[idx], y_tr[idx]

            optimizer.zero_grad()
            loss, components = cevae_loss_v2(nets, x_batch, t_batch, y_batch,
                                              beta_kl=beta_kl, w_recon_y=cfg.w_recon_y, w_aux_y=cfg.w_aux_y)
            loss.backward()
            optimizer.step()

            for k in epoch_sums:
                epoch_sums[k] += components[k]

        epoch_avg = {k: v / n_batches for k, v in epoch_sums.items()}
        epoch_avg["beta_kl"] = beta_kl
        history.append(epoch_avg)

        if verbose_every and (epoch % verbose_every == 0 or epoch == cfg.epochs - 1):
            c = epoch_avg
            print(f"epoch {epoch:4d}  beta_kl={c['beta_kl']:.2f}  total={c['total']:.3f}  "
                  f"recon_x={c['recon_x']:.3f} recon_t={c['recon_t']:.3f} recon_y={c['recon_y']:.3f} "
                  f"kl={c['kl']:.3f} aux_t={c['aux_t']:.3f} aux_y={c['aux_y']:.3f}")

    return history


def estimate_ite_v2(nets: dict, x):
    """Part5の estimate_ite を一般化したもの（nets辞書を受け取る版）。手順は設計書§4.4のまま。"""
    with torch.no_grad():
        t_prob = torch.sigmoid(nets["encoder_t"](x))
        t_hat = (t_prob > 0.5).float()

        y0_hat_q, y1_hat_q = nets["encoder_y"](x)
        y_hat = torch.where(t_hat == 1, y1_hat_q, y0_hat_q)

        mu_z, _ = nets["encoder_z"](x, t_hat, y_hat)

        y0_p, y1_p = nets["decoder_y"](mu_z)
        ite_pred = y1_p - y0_p
    return ite_pred


def diagnose_z_recovery(nets: dict, x, t, y, z_true) -> dict:
    """診断1を一般化したもの。encoder_z(x,t,y)のmu_zと真のZの相関を、符号反転を考慮して計算する。
    z_dimが2以上の場合は次元ごとに見て、絶対値最大のものを「z復元スコア」として返す
    （複数次元のうち1本だけが真のZ方向を捉え、他はノイズ、ということがあり得るため）。
    """
    with torch.no_grad():
        mu_z, _ = nets["encoder_z"](x, t, y)
    mu_z_np = mu_z.numpy()

    per_dim = []
    best_abs_corr = 0.0
    for k in range(mu_z_np.shape[1]):
        corr_pos = np.corrcoef(mu_z_np[:, k], z_true)[0, 1]
        per_dim.append((k, corr_pos))
        best_abs_corr = max(best_abs_corr, abs(corr_pos))

    return {"per_dim": per_dim, "best_abs_corr": best_abs_corr}


def evaluate_cevae(nets: dict, x_test, tau_true_test) -> dict:
    """テストセットでの ITE評価（√PEHE/MAE/eps_ATE）を返す。z復元度は別途 diagnose_z_recovery を使う。
    Part7のグリッドサーチで1設定ぶんの結果として使う。
    """
    tau_pred = estimate_ite_v2(nets, x_test).numpy()
    sqrt_pehe_v = np.sqrt(np.mean((tau_true_test - tau_pred) ** 2))
    mae_v = np.mean(np.abs(tau_true_test - tau_pred))
    eps_ate_v = abs(np.mean(tau_pred) - np.mean(tau_true_test))
    return {
        "sqrt_pehe": sqrt_pehe_v, "mae": mae_v, "eps_ate": eps_ate_v, "tau_pred": tau_pred,
    }

**動作確認**：今までの固定構成（z_dim=1, hidden=32, beta_kl_max=1.0）を、リファクタリングした関数群で
再現できるか確認する。ただし今回は **`df_train` だけで学習し `df_test` で評価**するように直す
（Part4はN=20000全部で学習していたので、ここからは数値が変わる可能性がある＝それ自体が意味のある比較になる）。

あわせて診断1の「z復元度」と、loss各項の推移（最後の数epoch）も出して、posterior collapse
（`kl`がほぼ0まで潰れていないか）が起きていないかも確認する。

In [ ]:
# train/testの分割に対応するZも作っておく（df_train/df_testはZ列を持たないため、indexで合わせる）
Z_series = pd.Series(Z, index=df_full.index)
Z_train = Z_series.loc[df_train.index].to_numpy()
Z_test = Z_series.loc[df_test.index].to_numpy()

x_train_tensor = torch.tensor(df_train[proxy_cols].values, dtype=torch.float32)
t_train_tensor = torch.tensor(df_train["T"].values, dtype=torch.float32)
y_train_tensor = torch.tensor(df_train["Y"].values, dtype=torch.float32)
x_test_tensor_v2 = torch.tensor(df_test[proxy_cols].values, dtype=torch.float32)
t_test_tensor_v2 = torch.tensor(df_test["T"].values, dtype=torch.float32)
y_test_tensor_v2 = torch.tensor(df_test["Y"].values, dtype=torch.float32)
tau_true_test_v2 = df_test["tau"].values

cfg_default = CEVAEConfig(z_dim=1, hidden=32, beta_kl_max=1.0, kl_anneal_epochs=75, epochs=150)
nets_default = build_cevae(cfg_default, d=d)
history_default = train_cevae(nets_default, cfg_default, x_train_tensor, t_train_tensor, y_train_tensor, verbose_every=30)

diag_default = diagnose_z_recovery(nets_default, x_train_tensor, t_train_tensor, y_train_tensor, Z_train)
eval_default = evaluate_cevae(nets_default, x_test_tensor_v2, tau_true_test_v2)

print()
print(f"z復元度（train, |corr|最大） = {diag_default['best_abs_corr']:.3f}")
print(f"CEVAE(df_trainのみ学習)  √PEHE={eval_default['sqrt_pehe']:.3f}  MAE={eval_default['mae']:.3f}  eps_ATE={eval_default['eps_ate']:.3f}")
print(f"X-learner（参考、再掲）  √PEHE={sqrt_pehe_x:.3f}  MAE={mae_x:.3f}  eps_ATE={eps_ate_x:.3f}")

## CEVAE Part 7：グリッドサーチ（z_dim × hidden × beta_kl_max）

Part6で作った `build_cevae` / `train_cevae` / `evaluate_cevae` / `diagnose_z_recovery` を使い、
次の3つの組み合わせを総当たりで試す：
```
z_dim_list       = [1, 2, 3]
hidden_list      = [32, 64, 128]
beta_kl_max_list = [0.1, 0.3, 0.5, 1.0]
```
3×3×4 = **36通り**。各設定ごとに `df_train` で学習 → `df_test` で評価し、
`√PEHE` / `MAE` / `eps_ATE` / `z復元度(|corr|最大)` を記録して表にする。

1設定あたり数十秒（epochs=150）かかるため、**36通り全部で10分前後**を見ておく。
（時間を短縮したい場合は `cfg_kwargs` の `epochs` を下げる、もしくは各リストを絞って試す）

**この表の読み方（切り分けの指針）**：
- `z復元度` が低いまま `√PEHE` も悪い → **zを掴めていないことがボトルネック**。z_dimやhidden、
  もっと根本的にはproxyの情報量（DGP側）を見直す必要がある。
- `z復元度` が高い（0.7〜0.8以上）のに `√PEHE` が悪い → **zは掴めているのにy/tau推定が悪い**。
  `decoder_y` のTARNetヘッドの外挿問題（Part5の結果メモ参照）が疑わしいので、
  `w_recon_y` を上げる・`weight_decay` を入れる等を試す方向。
- `beta_kl_max` を下げても `kl` が0近くまで潰れない／`z復元度`が変わらない →
  そもそも posterior collapse は起きていない（Part5の結果メモの結論と整合するはず）。
  逆に `beta_kl_max` を上げると`kl`がほぼ0になる設定があれば、それは collapse の兆候。

In [ ]:
z_dim_list = [1, 2, 3]
hidden_list = [32, 64, 128]
beta_kl_max_list = [0.1, 0.3, 0.5, 1.0]

grid_results = []

for z_dim_g, hidden_g, beta_kl_max_g in itertools.product(z_dim_list, hidden_list, beta_kl_max_list):
    cfg_g = CEVAEConfig(
        z_dim=z_dim_g, hidden=hidden_g, beta_kl_max=beta_kl_max_g,
        kl_anneal_epochs=75, epochs=150, batch_size=256, lr=1e-3, weight_decay=0.0, seed=0,
    )
    nets_g = build_cevae(cfg_g, d=d)
    train_cevae(nets_g, cfg_g, x_train_tensor, t_train_tensor, y_train_tensor, verbose_every=0)

    diag_g = diagnose_z_recovery(nets_g, x_train_tensor, t_train_tensor, y_train_tensor, Z_train)
    eval_g = evaluate_cevae(nets_g, x_test_tensor_v2, tau_true_test_v2)

    grid_results.append({
        "z_dim": z_dim_g, "hidden": hidden_g, "beta_kl_max": beta_kl_max_g,
        "sqrt_pehe": eval_g["sqrt_pehe"], "mae": eval_g["mae"], "eps_ate": eval_g["eps_ate"],
        "z_corr_abs": diag_g["best_abs_corr"],
    })
    r = grid_results[-1]
    print(f"z_dim={r['z_dim']} hidden={r['hidden']:3d} beta_kl_max={r['beta_kl_max']:.1f}  "
          f"√PEHE={r['sqrt_pehe']:6.3f}  MAE={r['mae']:6.3f}  eps_ATE={r['eps_ate']:5.3f}  z_corr={r['z_corr_abs']:.3f}")

df_grid = pd.DataFrame(grid_results).sort_values("sqrt_pehe").reset_index(drop=True)
df_grid

**X-learnerとの比較・ベスト設定の確認**：グリッド内の最良設定（√PEHE最小）が、それでも
X-learnerに勝てるかどうかを見る。勝てない場合は、Part8のDGP見直し（X-learnerが苦戦するように
proxyを難しくする・tauを非線形にする）と組み合わせて再実験する必要がある、という結論になる。

In [ ]:
best = df_grid.iloc[0]
print(f"グリッド内ベスト: z_dim={int(best['z_dim'])} hidden={int(best['hidden'])} beta_kl_max={best['beta_kl_max']}")
print(f"  CEVAE(ベスト)  √PEHE={best['sqrt_pehe']:.3f}  MAE={best['mae']:.3f}  eps_ATE={best['eps_ate']:.3f}  z_corr={best['z_corr_abs']:.3f}")
print(f"  X-learner      √PEHE={sqrt_pehe_x:.3f}  MAE={mae_x:.3f}  eps_ATE={eps_ate_x:.3f}")
print()
if best["sqrt_pehe"] < sqrt_pehe_x:
    print("→ この設定ではCEVAEがX-learnerに勝っている。")
else:
    print("→ この設定でもCEVAEはX-learnerに勝てていない。DGP側の見直し（Part8）が必要そう。")

## CEVAE Part 8：トイデータ生成の見直し案（v2、より公正な比較のため）

ここまでのトイデータ（## 0節）は、**XからZが線形でかなり良く復元できてしまう**設定だった
（corr(X_j,Z)が0.6〜0.8程度、`a_j ~ N(1.5,1.5)` のほぼ線形なローディング）。
これだとXGBoostベースのX-learnerが「Xを見ればほぼZが見えている」状態で戦えてしまい、
CEVAEが「複数proxyからZを復元する」という構造的な強みを発揮しにくい＝比較として不公平になりやすい。

設計書§0の言葉を借りると、これは「proxyが多く情報豊富 → ふつうの手法もX≈Zで調整できて差が消える」
の側に寄りすぎていた、ということ。狙いたいのは**中間のスイートスポット**：

1. **XからZが簡単に復元できすぎない**：Z→Xの間に非線形変換 `g(Z) = tanh(link_scale * Z)` を挟む。
   `tanh`は値を(-1,1)に飽和させるので、**線形回帰ではg(Z)の情報しか取り出せず、しかも非線形に歪んでいる**
   ため、単純な線形のproxy調整では取りこぼしが生じる。
2. **ただしCEVAEは推定できる余地を残す**：`tanh`は単調関数なので、MLP（Encoder）なら理論上は
   非線形に逆変換してZの情報を取り出せる可能性がある＝CEVAEの構造的優位性を検証する設定になる。
3. **T, Yへの交絡は変えない**：`T`, `tau`, `Y` は引き続き**真のZ**（`g(Z)`ではない）に依存させる。
   交絡の強さ自体は元のトイデータと同じにして、「proxyの質」だけを変える。
4. **tauを非線形にする**：`tau(Z) = alpha + beta*Z + delta_nl*(Z**2 - 1)` とする。
   `Z~N(0,1)`なら`E[Z**2]=1`なので`(Z**2-1)`は平均0＝ATE(=alpha)は変えずに、
   **異質性（ITEのばらつき）だけ非線形にする**トリック。単純な「Xの線形結合でtauを当てる」
   アプローチでは捉えにくくなる（ただしXGBoostは非線形当てはめ自体は得意なので、ここでの本質は
   「tauがZの非線形関数」というより「ZそのものをXから取り出すのが難しい」方が効いてくるはず）。

また、`d`（proxy本数）を増やし、ローディングを小さく・ノイズを大きくして、
**1本1本は弱い証拠だが、まとめれば情報がある**という、CEVAEの「複数proxyを統合する」前提に
合った状況を作る。

In [ ]:
def generate_toy_v2(N=20000, d=8, gamma=1.5, theta=2.0, alpha=1.0, beta=1.0, delta_nl=0.8,
                     sigma_x=3.5, sigma_y=1.0, link_scale=1.5, loading_mean=1.0, loading_std=1.0, seed=0):
    """より公正な比較のためのトイデータ生成（v2）。元の## 0節のトイデータとの違いはPart8のmarkdown参照。
    返り値は元のtoyデータと同じ並び: X, T, Y, Y0, Y1, tau, Z
    """
    rng = np.random.default_rng(seed)
    Z_v2 = rng.normal(0, 1, size=N)
    g_z = np.tanh(link_scale * Z_v2)                      # (-1,1)に飽和する非線形変換

    a_v2 = rng.normal(loading_mean, loading_std, size=d)
    X_v2 = g_z[:, None] * a_v2[None, :] + rng.normal(0, sigma_x, size=(N, d))

    T_v2 = rng.binomial(1, sigmoid(gamma * Z_v2))           # 交絡は「真のZ」に対して効かせる（gではない）

    tau_v2 = alpha + beta * Z_v2 + delta_nl * (Z_v2 ** 2 - 1)   # 非線形ITE（ATEはalphaのまま）
    base_v2 = theta * Z_v2
    Y0_v2 = base_v2 + rng.normal(0, sigma_y, size=N)
    Y1_v2 = base_v2 + tau_v2 + rng.normal(0, sigma_y, size=N)
    Y_v2 = np.where(T_v2 == 1, Y1_v2, Y0_v2)

    return X_v2, T_v2, Y_v2, Y0_v2, Y1_v2, tau_v2, Z_v2


**動作確認**：狙い通りの性質になっているか診断する。具体的に見るのは
- `corr(T,Z)`：交絡の強さが元のトイデータと同程度に保たれているか
- `R²(Z|X)`（線形回帰）：元のトイデータ（0.6〜0.8程度の相関）より**下がっているか**
  （＝線形では復元しにくくなっているか）
- `corr(tau, Z)` と `corr(tau, Z**2)`：tauが線形成分・非線形成分の両方を持っているか
- ナイーブATEのバイアス：交絡が効いていれば、依然として真のATE(=alpha)から大きくズレるはず

In [ ]:
X2, T2, Y2, Y0_2, Y1_2, tau2, Z2 = generate_toy_v2()

true_ate2 = tau2.mean()
naive_ate2 = Y2[T2 == 1].mean() - Y2[T2 == 0].mean()
corr_t_z2 = np.corrcoef(T2, Z2)[0, 1]

# 全proxyを使った線形回帰でどこまでZを復元できるか（元のトイデータと同じ診断）
Xd2 = np.column_stack([np.ones(len(Z2)), X2])
coef2, _, _, _ = np.linalg.lstsq(Xd2, Z2, rcond=None)
pred_z2 = Xd2 @ coef2
r2_z2 = 1 - np.sum((Z2 - pred_z2) ** 2) / np.sum((Z2 - Z2.mean()) ** 2)

corr_tau_z = np.corrcoef(tau2, Z2)[0, 1]
corr_tau_z2sq = np.corrcoef(tau2, Z2 ** 2)[0, 1]

print(f"真のATE                    : {true_ate2:.3f}")
print(f"ナイーブATE(差の平均)      : {naive_ate2:.3f}   ← 交絡バイアス = {naive_ate2 - true_ate2:+.3f}")
print(f"corr(T, Z)                 : {corr_t_z2:.3f}  （元のトイデータと同程度の交絡強度になっているか）")
print(f"線形回帰での R²(Z|X)       : {r2_z2:.3f}  （元のトイデータは0.6〜0.8程度の相関 → これより下がっているか）")
print(f"corr(tau, Z)                : {corr_tau_z:.3f}  （線形成分）")
print(f"corr(tau, Z**2)             : {corr_tau_z2sq:.3f}  （非線形成分。0でなければ非線形性が乗っている）")

**v2データへの差し替え方**：`## 0節`〜Part7のパイプラインは変数名 `X, T, Y, tau, Z`（および
それらから作る `df_full, df_train, df_test, proxy_cols`）に依存している。v2で再実験したい場合は、
```python
X, T, Y, Y0, Y1, tau, Z = generate_toy_v2()
d = X.shape[1]
```
のように上書きしてから、`proxy_cols = [f"X{j+1}" for j in range(d)]` 以降（X-learnerの
`df_full/df_train/df_test` 構築）→ Part6/Part7のセルを**上から順に再実行**すればよい
（クラス定義やヘルパー関数はそのまま使い回せる。`d`が変わるので `build_cevae` 等は
毎回 `d=d` を渡す作りに既にしてある）。

## CEVAE Part 9：CEVAE と X-learner、どちらが有利になりやすいか（まとめ）

今回の一連の診断（診断1, Part6〜8）から分かったことを整理する。

### 今回の失敗の原因切り分け（再掲＋更新）
- **「zを掴めていない」のか「zは掴めているがy/tau推定が悪いのか」**：
  元の設定（z_dim=1, hidden=32, beta_kl_max=1.0, N全部で学習）では `corr(mu_z,Z)≈0.47` と
  z復元は中途半端だった。Part7のグリッドサーチの**速報（4設定だけのスモークテスト、epoch数も20に
  縮小した暫定結果なので参考値）**では、`z_dim=2` にしただけで `z復元度が0.92〜0.94` まで上がり、
  `√PEHE` も大きく改善した（z_dim=1: √PEHE≈7〜9 → z_dim=2: √PEHE≈1.3〜2.7）。
  → **z_dim=1 という設計（真のZがスカラーだから1で十分、という思い込み）自体がボトルネックだった**
  可能性が高い。本番の36設定フルグリッドで確認する価値がある。
- **KL過剰によるposterior collapse**：`beta_kl` を最大1.0まで上げても `kl` は0近くまで潰れておらず
  （2.2前後で安定）、collapseは起きていなかった。スモークテストでも `beta_kl_max=1.0` の方が
  `0.3` より良い結果だった＝**KLを抑えすぎる方が今回はむしろ逆効果**だった可能性がある。
- **decoder_yの反実仮想外挿**（Part5の結果メモ）：これは今も残っている課題。z_dim/hidden/beta_klの
  調整である程度緩和されたが、根本対策（weight_decay、出力のクリップ等）は未検証。

### proxyの質（R²(Z|X)）の影響
元のトイデータは全proxyの線形回帰で `R²(Z|X)≈0.81`、つまり**Xを見ればほぼZが見えている**状態
だった。Part8のv2（非線形リンク `tanh` + proxy本数を増やしてノイズも増やす）では `R²(Z|X)≈0.45` まで
下がり、線形では復元しにくい一方、交絡強度（`corr(T,Z)`）やtauの非線形性は保ったまま、という
「CEVAEにとってよりフェアな」設定に近づけた（実際にCEVAE/X-learnerを再実験するのは次回のタスク）。

### 一般的な傾向（設計書§0の整理＋今回の実験から得た補足）

| 条件 | 有利になりやすい手法 | 理由 |
|---|---|---|
| proxyが少ない／ノイズが極端に大きい | **どちらも不利**（CEVAEも復元できない） | CEVAEの「複数proxyからZを復元する」前提が成立しない |
| proxyが少数で、かつXがほぼ線形にZを再現する（R²(Z|X)が高い） | **X-learner**（今回の元のトイデータがこれ） | XGBoostがXから直接tauを学習でき、Zを明示的に復元する手間が要らない。CEVAEが学習不安定なぶん不利 |
| proxyが多数・個々は弱いが非線形/redundantな関係でZを反映 | **CEVAE（が安定して学習できれば）** | 「複数proxyの背後に共通の低次元Zがある」という構造的仮定が活き、ナイーブな線形/木ベース調整では拾いきれない情報を統合できる |
| CEVAEの学習が不安定（z_dim過小・正則化なし・epoch不足等） | **X-learner**（CEVAEの理論的優位が実装の不安定さに負ける） | 今回の元の結果（z_dim=1, weight_decay=0）がまさにこのケース |
| サンプルサイズが小さい | **X-learner寄り**（木ベースの方がニューラルより少データで安定しやすい） | CEVAEはニューラル+VAEで学習が不安定になりやすく、データが少ないとさらに悪化する |

### 結論・次のアクション
1. **Part7のフルグリッドサーチ（36設定、epoch=150）を実際に回す**：z_dim=2/3が本当に有効か、
   beta_kl_maxはどの値が良いかを確定させる。
2. **Part8のv2 DGPで同じ実験を繰り返す**：proxyの質を下げてもCEVAEが（ベストな設定の下で）
   X-learnerに勝てるかを見る。これが「CEVAEのスイートスポット」を実証する本実験の核心になる。
3. **decoder_yの外挿問題への対策**（weight_decay、出力クリップ等）は、上2つの結果を見てから
   優先度を判断する（z_dim/proxyの質の影響の方が大きい可能性があるため）。
4. 本番の2次元DGP（`DGPConfig`, §2.1）に進む際も、同じ考え方（proxyの質・z_dim・正則化）を
   引き継いで設計する。

## 0c. v2データで再実験（proxyを難しくして、X-learnerとの比較をより意味のあるものにする）

元のトイデータは R²(Z|X)≈0.81 と「Xを見ればほぼZが見えている」状態だったため、
X-learnerが易しすぎた（√PEHE=0.844）。ここからは Part8 で定義した `generate_toy_v2()` に差し替えて、
**R²(Z|X)≈0.45（線形では復元しにくい）かつtauが非線形という、よりフェアな設定**で同じ実験を行う。

このセル以降を上から順に実行すると：
1. `X, T, Y, tau, Z, d` をv2データで上書き
2. `proxy_cols / df_full / df_train / df_test` を新しい `d=8` で再構築
3. X-learner を再フィット（ハイパラ探索あり、数分かかる）
4. Part6用テンソルを再構築
5. Part7のグリッドサーチセルを再実行（そのまま使える）

In [ ]:
# v2データで X, T, Y, Y0, Y1, tau, Z, d を上書きする
X, T, Y, Y0, Y1, tau, Z = generate_toy_v2()
d = X.shape[1]   # 8に変わる

# 簡単な確認
print(f"d={d}, N={len(Z)}, tau 平均={tau.mean():.3f}（≈alpha=1）")
print(f"ナイーブATE={Y[T==1].mean()-Y[T==0].mean():.3f}  真のATE={tau.mean():.3f}  バイアス={Y[T==1].mean()-Y[T==0].mean()-tau.mean():+.3f}")
Xd_v2 = np.column_stack([np.ones(len(Z)), X])
coef_v2, _, _, _ = np.linalg.lstsq(Xd_v2, Z, rcond=None)
r2_v2 = 1 - np.sum((Z - Xd_v2 @ coef_v2)**2) / np.sum((Z - Z.mean())**2)
print(f"R²(Z|X)線形回帰={r2_v2:.3f}  （元のトイデータは0.81。大幅に低下しているか）")

In [ ]:
# proxy_cols / df_full / df_train / df_test を v2データ(d=8)で再構築
proxy_cols = [f"X{j+1}" for j in range(d)]
df_full = pd.DataFrame(X, columns=proxy_cols)
df_full["T"] = T
df_full["Y"] = Y
df_full["tau"] = tau

df_train, df_test = train_test_split(df_full, test_size=0.3, random_state=0, stratify=df_full["T"])
print(f"df_train: {len(df_train)}, df_test: {len(df_test)}")

In [ ]:
# X-learner を v2 データで再フィット（ハイパラ探索あり、数分かかる）
df0_v2 = df_train[df_train["T"] == 0]
df1_v2 = df_train[df_train["T"] == 1]

print("ハイパラ探索中（n_iter=50, cv=5）...")
t0_params_v2 = search_xgb_params(df0_v2[proxy_cols], df0_v2["Y"])
t1_params_v2 = search_xgb_params(df1_v2[proxy_cols], df1_v2["Y"])
t0_model_v2 = XGBRegressor(**t0_params_v2).fit(df0_v2[proxy_cols], df0_v2["Y"])
t1_model_v2 = XGBRegressor(**t1_params_v2).fit(df1_v2[proxy_cols], df1_v2["Y"])

D0_v2 = t1_model_v2.predict(df0_v2[proxy_cols]) - df0_v2["Y"].values
D1_v2 = df1_v2["Y"].values - t0_model_v2.predict(df1_v2[proxy_cols])
x0_params_v2 = search_xgb_params(df0_v2[proxy_cols], D0_v2)
x1_params_v2 = search_xgb_params(df1_v2[proxy_cols], D1_v2)
x0_model_v2 = XGBRegressor(**x0_params_v2).fit(df0_v2[proxy_cols], D0_v2)
x1_model_v2 = XGBRegressor(**x1_params_v2).fit(df1_v2[proxy_cols], D1_v2)

prop_model_v2 = LogisticRegression(max_iter=1000).fit(df_train[proxy_cols], df_train["T"])
g0_test_v2 = prop_model_v2.predict_proba(df_test[proxy_cols])[:, 0]
tau0_test_v2 = x0_model_v2.predict(df_test[proxy_cols])
tau1_test_v2 = x1_model_v2.predict(df_test[proxy_cols])
tau_pred_x_v2 = g0_test_v2 * tau0_test_v2 + (1 - g0_test_v2) * tau1_test_v2

tau_true_test_x = df_test["tau"].values
sqrt_pehe_x = np.sqrt(np.mean((tau_true_test_x - tau_pred_x_v2) ** 2))
mae_x = np.mean(np.abs(tau_true_test_x - tau_pred_x_v2))
eps_ate_x = abs(np.mean(tau_pred_x_v2) - np.mean(tau_true_test_x))
print(f"X-learner(v2)  √PEHE={sqrt_pehe_x:.3f}  MAE={mae_x:.3f}  eps_ATE={eps_ate_x:.3f}")

In [ ]:
# Part6のテンソルを v2 データで再構築（この後 Part7 グリッドサーチをそのまま再実行できる）
Z_series = pd.Series(Z, index=df_full.index)
Z_train = Z_series.loc[df_train.index].to_numpy()
Z_test  = Z_series.loc[df_test.index].to_numpy()

x_train_tensor    = torch.tensor(df_train[proxy_cols].values, dtype=torch.float32)
t_train_tensor    = torch.tensor(df_train["T"].values,        dtype=torch.float32)
y_train_tensor    = torch.tensor(df_train["Y"].values,        dtype=torch.float32)
x_test_tensor_v2  = torch.tensor(df_test[proxy_cols].values,  dtype=torch.float32)
t_test_tensor_v2  = torch.tensor(df_test["T"].values,         dtype=torch.float32)
y_test_tensor_v2  = torch.tensor(df_test["Y"].values,         dtype=torch.float32)
tau_true_test_v2  = df_test["tau"].values

print("テンソル再構築完了")
print(f"x_train_tensor: {tuple(x_train_tensor.shape)}  x_test_tensor_v2: {tuple(x_test_tensor_v2.shape)}")
print()
print("▼ この後 Part7 グリッドサーチ（既存セル）をそのまま再実行してください")
print(f"  X-learner(v2)の比較ベース: √PEHE={sqrt_pehe_x:.3f}")

## 1. DGP（人工データ生成）

設計書 §2.1 の生成過程を実装。去年の卒論コードの列名規約に合わせる:
```
W   = 処置 (treatment, 0/1)
tau = 真のITE
Y0, Y1 = 潜在結果
X1..Xd = 観測される proxy
```

★重要: X は Z の proxy（X = Z@a + ノイズ）であって交絡因子そのものではない。
モデルに渡すのは X, W, Y のみ。Z は Oracle 以外に絶対渡さない。

★Z は2次元 (z1, z2)。proxy ローディング a は (2, d) の行列で、
実験seed（Z・ノイズ・Wの乱数）とは別の a_seed で生成し、sweep全体で固定する
（proxy の構造そのものが変わると γ や σx の効果と交絡してしまうため）。

DAG:
```
        Z=(z1,z2)   (潜在交絡, 未観測)
      /    |    \
     X     W  →  Y
```


In [8]:
@dataclass
class DGPConfig:
    N: int = 20000          # サンプル数
    d: int = 10              # proxy の本数（Z を覗く窓の数）
    sigma_x: float = 1.0     # proxy ノイズ標準偏差（Sweep B のノブ）。0で X=Z@a（完全観測）
    gamma: float = 1.5       # Z→W 交絡強度（Sweep A のノブ）。0で交絡なし(RCT)
    theta: np.ndarray = field(default_factory=lambda: np.array([2.0, 2.0]))  # Z→Y 交絡強度（長さ2）
    alpha: float = 1.0       # ITE の平均 (= 真のATE)
    beta: np.ndarray = field(default_factory=lambda: np.array([1.0, 1.0]))   # ITE の異質性（長さ2、0で均質効果）
    mu0: float = 0.0         # 結果のベース水準
    sigma_y: float = 0.5     # 結果ノイズ標準偏差
    a: np.ndarray = None     # proxy ローディング行列 (2, d)。Noneなら a_seed から生成して固定
    a_seed: int = 12345      # a 専用の乱数シード（実験seedと分離し、sweep間で proxy 構造を固定する）
    seed: int = 0            # データ生成（Z, ノイズ, W）の乱数シード


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def _get_loading(cfg: DGPConfig) -> np.ndarray:
    """proxy ローディング行列 a (shape (2, d)) を返す。
    cfg.a が指定されていればそれを使い、なければ a_seed から生成する。
    """
    if cfg.a is not None:
        a = np.asarray(cfg.a, dtype=float)
        assert a.shape == (2, cfg.d), f"a の形は (2,{cfg.d}) である必要があります（実際: {a.shape}）"
        return a
    return np.random.default_rng(cfg.a_seed).normal(0.0, 1.0, size=(2, cfg.d))


def _linear_r2(X: np.ndarray, target: np.ndarray) -> float:
    """X（全proxy）から target を線形回帰した場合の R²。proxyがZをどれだけ線形復元できるかの目安。"""
    Xd = np.column_stack([np.ones(len(target)), X])
    coef, _, _, _ = np.linalg.lstsq(Xd, target, rcond=None)
    pred = Xd @ coef
    ss_res = np.sum((target - pred) ** 2)
    ss_tot = np.sum((target - target.mean()) ** 2)
    return 1.0 - ss_res / ss_tot


def generate(cfg: DGPConfig):
    """
    DGPConfig に従って人工データを生成する。

    Returns (すべて長さ N。Z のみ (N,2)):
        X   : (N, d) 観測 proxy
        W   : (N,)   処置 0/1
        Y   : (N,)   観測結果 Y(W)
        Y0  : (N,)   潜在結果 Y(0)
        Y1  : (N,)   潜在結果 Y(1)
        tau : (N,)   真のITE = alpha + beta・Z
        Z   : (N,2)  潜在交絡（評価・Oracle 専用。学習には使わない）
    """
    rng = np.random.default_rng(cfg.seed)
    theta = np.asarray(cfg.theta, dtype=float)
    beta = np.asarray(cfg.beta, dtype=float)
    a = _get_loading(cfg)

    # 1) 潜在交絡 Z ~ N(0, I_2)
    Z = rng.normal(0.0, 1.0, size=(cfg.N, 2))

    # 2) proxy  X = Z @ a + e,  e ~ N(0, sigma_x^2)
    eps = rng.normal(0.0, cfg.sigma_x, size=(cfg.N, cfg.d))
    X = Z @ a + eps                                          # (N, d)

    # 3) 処置 W ~ Bernoulli(sigmoid(gamma * (z1+z2)/√2))  ← 交絡: 割り当てが Z に依存
    z_dir = (Z[:, 0] + Z[:, 1]) / np.sqrt(2.0)
    propensity = _sigmoid(cfg.gamma * z_dir)
    W = rng.binomial(1, propensity)

    # 4) 結果 Y(t) = mu0 + theta・Z + t*(alpha + beta・Z) + noise
    tau = cfg.alpha + Z @ beta                                # 真のITE（Z に対し異質）
    base = cfg.mu0 + Z @ theta                                # 交絡項（T=0 のときの水準）
    eta0 = rng.normal(0.0, cfg.sigma_y, size=cfg.N)
    eta1 = rng.normal(0.0, cfg.sigma_y, size=cfg.N)
    Y0 = base + eta0
    Y1 = base + tau + eta1
    Y = np.where(W == 1, Y1, Y0)

    return X, W, Y, Y0, Y1, tau, Z


def make_df(X, W, Y, Y0, Y1, tau, Z=None):
    """去年の make_df 流儀。Z1, Z2 は Oracle 用に保持（学習時は drop する）。"""
    d = X.shape[1]
    df = pd.DataFrame(X, columns=[f"X{i+1}" for i in range(d)])
    df["W"] = W
    df["Y"] = Y
    df["Y0"] = Y0
    df["Y1"] = Y1
    df["tau"] = tau
    cols = ["tau", "W", "Y", "Y0", "Y1"] + [f"X{i+1}" for i in range(d)]
    if Z is not None:
        df["Z1"] = Z[:, 0]
        df["Z2"] = Z[:, 1]
        cols += ["Z1", "Z2"]
    return df[cols]


In [9]:
def diagnostics(cfg: DGPConfig):
    """このcfgが「交絡あり・proxyが意味を持つ」設定になっているか健全性チェック。"""
    X, W, Y, Y0, Y1, tau, Z = generate(cfg)

    true_ate = tau.mean()
    # ナイーブ推定（差の平均）: 交絡があると真のATEからズレる
    naive_ate = Y[W == 1].mean() - Y[W == 0].mean()

    z_dir = (Z[:, 0] + Z[:, 1]) / np.sqrt(2.0)
    corr_w_zdir = np.corrcoef(W, z_dir)[0, 1]

    # proxy が z1, z2 をそれぞれどれだけ説明できているか（1本あたりの相関）
    corr_xz1 = np.array([np.corrcoef(X[:, j], Z[:, 0])[0, 1] for j in range(cfg.d)])
    corr_xz2 = np.array([np.corrcoef(X[:, j], Z[:, 1])[0, 1] for j in range(cfg.d)])

    # 全proxyを使った線形回帰でどこまでZを復元できるか（情報量の目安）
    r2_z1 = _linear_r2(X, Z[:, 0])
    r2_z2 = _linear_r2(X, Z[:, 1])

    print(f"--- DGP diagnostics (gamma={cfg.gamma}, sigma_x={cfg.sigma_x}, d={cfg.d}, N={cfg.N}) ---")
    print(f"処置割合 P(W=1)            : {W.mean():.3f}")
    print(f"真のATE (=alpha)           : {true_ate:.3f}")
    print(f"ナイーブATE(差の平均)      : {naive_ate:.3f}   ← 交絡バイアス = {naive_ate - true_ate:+.3f}")
    print(f"corr(W, (z1+z2)/√2)        : {corr_w_zdir:.3f}   (0なら交絡なし)")
    print(f"ITEの範囲                  : [{tau.min():.2f}, {tau.max():.2f}]  (異質性 std={tau.std():.2f})")
    print(f"proxy相関 corr(X_j, z1)    : {np.round(corr_xz1, 3)}")
    print(f"proxy相関 corr(X_j, z2)    : {np.round(corr_xz2, 3)}")
    print(f"全proxy線形回復 R²(z1|X)   : {r2_z1:.3f}   (1に近いほどXからz1を完全復元できる)")
    print(f"全proxy線形回復 R²(z2|X)   : {r2_z2:.3f}")


In [10]:
# 基準設定でのサニティチェック
diagnostics(DGPConfig())

# 参考: 交絡を切った場合（去年と同じRCT相当）はナイーブATEがほぼ真値に一致するはず
print()
diagnostics(DGPConfig(gamma=0.0, theta=np.zeros(2)))


--- DGP diagnostics (gamma=1.5, sigma_x=1.0, d=10, N=20000) ---
処置割合 P(W=1)            : 0.505
真のATE (=alpha)           : 1.005
ナイーブATE(差の平均)      : 4.738   ← 交絡バイアス = +3.734
corr(W, (z1+z2)/√2)        : 0.533   (0なら交絡なし)
ITEの範囲                  : [-4.41, 6.95]  (異質性 std=1.41)
proxy相関 corr(X_j, z1)    : [-0.493  0.669 -0.566 -0.197 -0.063 -0.599 -0.731  0.376  0.295 -0.754]
proxy相関 corr(X_j, z2)    : [ 0.804  0.511 -0.487  0.655 -0.414 -0.041  0.428 -0.727  0.477  0.542]
全proxy線形回復 R²(z1|X)   : 0.896   (1に近いほどXからz1を完全復元できる)
全proxy線形回復 R²(z2|X)   : 0.905

--- DGP diagnostics (gamma=0.0, sigma_x=1.0, d=10, N=20000) ---
処置割合 P(W=1)            : 0.494
真のATE (=alpha)           : 1.005
ナイーブATE(差の平均)      : 0.993   ← 交絡バイアス = -0.011
corr(W, (z1+z2)/√2)        : 0.004   (0なら交絡なし)
ITEの範囲                  : [-4.41, 6.95]  (異質性 std=1.41)
proxy相関 corr(X_j, z1)    : [-0.493  0.669 -0.566 -0.197 -0.063 -0.599 -0.731  0.376  0.295 -0.754]
proxy相関 corr(X_j, z2)    : [ 0.804  0.511 -0.487  0.655 -0.414

## 2. 評価指標（metrics）

設計書 §5。去年の卒論コードの RMSE/MAE を流用し、ε_ATE を追加。

PEHE = √PEHE の二乗 = ITE推定のMSE。√PEHE が因果推論の標準報告値なので
`sqrt_pehe` という名前にしている（去年の RMSE と同じ計算）。


In [11]:
def pehe(tau_true, tau_pred):
    return np.mean((np.asarray(tau_true) - np.asarray(tau_pred)) ** 2)


def sqrt_pehe(tau_true, tau_pred):
    return np.sqrt(pehe(tau_true, tau_pred))


def mae(tau_true, tau_pred):
    return np.mean(np.abs(np.asarray(tau_true) - np.asarray(tau_pred)))


def eps_ate(tau_true, tau_pred):
    return abs(np.mean(tau_pred) - np.mean(tau_true))


def summarize(tau_true, tau_pred) -> dict:
    return {
        "sqrt_pehe": sqrt_pehe(tau_true, tau_pred),
        "mae": mae(tau_true, tau_pred),
        "eps_ate": eps_ate(tau_true, tau_pred),
    }


## 3. ベースライン：Naive / S-learner / T-learner / X-learner / Oracle

去年の卒論コード（`claude_cowork/研究室/2025年度/卒論/卒論python.py`）の
S/T/X-learner実装を整理移植（設計書 §13）。

去年からの変更点（実装方針として確定済み）：
- DGP は上の新しいもの（潜在交絡 Z あり）。学習に渡すのは X1..Xd, W, Y のみ。
  Z は Oracle 専用（§1の方針：絶対に通常モデルへ渡さない）。
- base learner は **XGBoost のみ**（RandomForestは省略）。
- ハイパーパラメータ探索（RandomizedSearchCV）は **γ/σx の設定ごとに1回だけ**
  代表データで実行し、その設定の全seedで固定パラメータを再利用する
  （N=20000 × 20 seeds × 5 γ点 を毎回フルサーチすると非現実的な時間になるため）。
  → `tune_hyperparams()` で1回探索 → `run_all()` に固定パラメータを渡して全seedで再利用。
- Oracle は「T-learner にそのまま Z を渡すだけ」（設計書 §3）。
  T-learner と同じハイパーパラメータを使い、入力特徴量だけ X1..Xd → Z1,Z2 に差し替える。
- X-learner の傾向スコア合成は去年のまま：
  propensity = P(W=0|x)（LogisticRegression）, tau = tau0*propensity + tau1*(1-propensity)


In [12]:
# 去年の param_distributions をそのまま流用
XGB_PARAM_DIST = {
    "n_estimators": randint(300, 2500),
    "learning_rate": uniform(0.01, 0.19),
    "max_depth": randint(2, 9),
    "min_child_weight": randint(1, 10),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "reg_lambda": uniform(0.0, 10.0),
}


def feature_cols(d: int) -> list:
    return [f"X{i+1}" for i in range(d)]


def search_xgb_params(X, y, n_iter=50, cv=5, seed=0) -> dict:
    """RandomizedSearchCVでXGBRegressorの最良パラメータを探索して返す。
    γ/σxの設定ごとに1回だけ呼び、結果を同設定の全seedで再利用する想定。
    """
    search = RandomizedSearchCV(
        XGBRegressor(objective="reg:squarederror", random_state=seed, verbosity=0),
        param_distributions=XGB_PARAM_DIST,
        n_iter=n_iter,
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
        random_state=seed,
    )
    search.fit(X, y)
    return search.best_estimator_.get_params()


In [13]:
@dataclass
class BaselineHyperparams:
    """S/T0/T1/X0/X1 サブモデルのXGBoostパラメータ（γ設定ごとに1回探索して固定）。"""
    s: dict
    t0: dict
    t1: dict
    x0: dict
    x1: dict


def tune_hyperparams(df_train: pd.DataFrame, d: int, n_iter=50, cv=5, seed=0) -> BaselineHyperparams:
    """代表データ(df_train)を使い、S/T0/T1/X0/X1のXGBハイパーパラメータを一度だけ探索する。"""
    feat = feature_cols(d)

    s_params = search_xgb_params(df_train[feat + ["W"]], df_train["Y"], n_iter=n_iter, cv=cv, seed=seed)

    df0 = df_train[df_train["W"] == 0]
    df1 = df_train[df_train["W"] == 1]
    t0_params = search_xgb_params(df0[feat], df0["Y"], n_iter=n_iter, cv=cv, seed=seed)
    t1_params = search_xgb_params(df1[feat], df1["Y"], n_iter=n_iter, cv=cv, seed=seed)

    # X-learner の疑似ITE（D0, D1）を作るために、探索済みのT-learnerを一度fitする
    t0_model = XGBRegressor(**t0_params).fit(df0[feat], df0["Y"])
    t1_model = XGBRegressor(**t1_params).fit(df1[feat], df1["Y"])
    D0 = t1_model.predict(df0[feat]) - df0["Y"].values
    D1 = df1["Y"].values - t0_model.predict(df1[feat])
    x0_params = search_xgb_params(df0[feat], D0, n_iter=n_iter, cv=cv, seed=seed)
    x1_params = search_xgb_params(df1[feat], D1, n_iter=n_iter, cv=cv, seed=seed)

    return BaselineHyperparams(s=s_params, t0=t0_params, t1=t1_params, x0=x0_params, x1=x1_params)


In [14]:
def run_all(df_train: pd.DataFrame, df_test: pd.DataFrame, d: int, hp: BaselineHyperparams) -> dict:
    """固定パラメータ hp でNaive/S/T/X/(Oracle)を学習し、test の ITE 予測を返す。
    1seed分のデータ生成のたびに呼ぶ（ハイパーパラメータ探索はしない＝高速パス）。
    """
    feat = feature_cols(d)
    results = {}

    # Naive: 差の平均（train で推定し、test 全行に同じ値を予測）
    ate_hat = df_train.loc[df_train["W"] == 1, "Y"].mean() - df_train.loc[df_train["W"] == 0, "Y"].mean()
    results["naive"] = np.full(len(df_test), ate_hat)

    # S-learner: X+W を入力にした単一モデル
    s_model = XGBRegressor(**hp.s).fit(df_train[feat + ["W"]], df_train["Y"])
    test_w0 = df_test[feat].copy(); test_w0["W"] = 0
    test_w1 = df_test[feat].copy(); test_w1["W"] = 1
    mu0_s = s_model.predict(test_w0[feat + ["W"]])
    mu1_s = s_model.predict(test_w1[feat + ["W"]])
    results["s"] = mu1_s - mu0_s

    # T-learner: W=0/1で別モデル（X-learnerでも再利用する）
    df0 = df_train[df_train["W"] == 0]
    df1 = df_train[df_train["W"] == 1]
    t0_model = XGBRegressor(**hp.t0).fit(df0[feat], df0["Y"])
    t1_model = XGBRegressor(**hp.t1).fit(df1[feat], df1["Y"])
    results["t"] = t1_model.predict(df_test[feat]) - t0_model.predict(df_test[feat])

    # X-learner: T-learnerの疑似ITEを回帰し、傾向スコアで合成
    D0 = t1_model.predict(df0[feat]) - df0["Y"].values
    D1 = df1["Y"].values - t0_model.predict(df1[feat])
    x0_model = XGBRegressor(**hp.x0).fit(df0[feat], D0)
    x1_model = XGBRegressor(**hp.x1).fit(df1[feat], D1)
    prop_model = LogisticRegression(max_iter=1000).fit(df_train[feat], df_train["W"])
    g0_test = prop_model.predict_proba(df_test[feat])[:, 0]  # P(W=0|x)
    tau0_test = x0_model.predict(df_test[feat])
    tau1_test = x1_model.predict(df_test[feat])
    results["x"] = g0_test * tau0_test + (1 - g0_test) * tau1_test

    # Oracle: T-learnerと全く同じ構成・同じハイパーパラメータで、入力だけ Z に差し替え
    if "Z1" in df_train.columns and "Z2" in df_train.columns:
        zc = ["Z1", "Z2"]
        z0_model = XGBRegressor(**hp.t0).fit(df0[zc], df0["Y"])
        z1_model = XGBRegressor(**hp.t1).fit(df1[zc], df1["Y"])
        results["oracle"] = z1_model.predict(df_test[zc]) - z0_model.predict(df_test[zc])

    return results


### 動作確認（スモークテスト）

本番のsweepは N=20000・n_iter=50 で別途実行する。ここでは縮小設定（N=3000, n_iter=10, cv=3）で
コードが正しく動くかだけを確認する。


In [15]:
cfg = DGPConfig(N=3000)
X, W, Y, Y0, Y1, tau, Z = generate(cfg)
df = make_df(X, W, Y, Y0, Y1, tau, Z)
df_train, df_test = train_test_split(df, test_size=0.3, random_state=0, stratify=df["W"])

print("ハイパーパラメータ探索中（スモークテスト用に n_iter=10, cv=3 に縮小）...")
hp = tune_hyperparams(df_train, cfg.d, n_iter=10, cv=3, seed=0)

print("固定パラメータで学習・予測中...")
preds = run_all(df_train, df_test, cfg.d, hp)
tau_true = df_test["tau"].values

print(f"\n--- スモークテスト結果 (N={cfg.N}, gamma={cfg.gamma}) ---")
for name, tau_pred in preds.items():
    m = summarize(tau_true, tau_pred)
    print(f"{name:8s} √PEHE={m['sqrt_pehe']:.3f}  MAE={m['mae']:.3f}  eps_ATE={m['eps_ate']:.3f}")


ハイパーパラメータ探索中（スモークテスト用に n_iter=10, cv=3 に縮小）...


固定パラメータで学習・予測中...



--- スモークテスト結果 (N=3000, gamma=1.5) ---
naive    √PEHE=4.045  MAE=3.824  eps_ATE=3.822
s        √PEHE=1.112  MAE=0.933  eps_ATE=0.846
t        √PEHE=1.465  MAE=1.193  eps_ATE=1.009
x        √PEHE=1.367  MAE=1.126  eps_ATE=1.016
oracle   √PEHE=0.501  MAE=0.357  eps_ATE=0.132
